<div style="
    width:100%;
    max-width:1080px;
    margin:24px 0 32px 0;
    background:linear-gradient(180deg, #1f2937 0%, #0f172a 100%);
    border:1px solid #334155;
    border-radius:22px;
    box-shadow:0 18px 38px rgba(15,23,42,0.24);
    overflow:hidden;
    color:#e6edf7;
">

<div style="
    height:8px;
    background:linear-gradient(90deg, #c2410c 0%, #f59e0b 100%);
"></div>

<div style="
    max-width:860px;
    padding:34px 38px 36px 38px;
    margin-left:0;
    margin-right:auto;
    font-family:'Aptos', 'Segoe UI Variable', 'Segoe UI', sans-serif;
    font-size:19px;
    line-height:1.78;
    color:#e6edf7;
">

<div style="margin-bottom:22px;">
<h1 style="margin:0 0 14px 0; font-size:34px; line-height:1.2; font-weight:700; color:#ffffff;">El Fine Tuning para la Clasificación de Texto</h1>
<p style="margin:0; color:#cbd5e1; font-size:18px;">Este notebook reúne los componentes anteriores en una tarea concreta de clasificación de texto mediante un modelo preentrenado ajustado con Trainer. Aquí la teoría se convierte en pipeline aplicado.</p>
</div>

<div style="display:grid; gap:14px;">
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Por qué importa</div>
    <div style="color:#f8fafc;">Permite ver cómo tokenización, dataset, modelo y configuración se integran en un flujo completo de NLP moderno.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Qué vas a aprender aquí</div>
    <div style="color:#f8fafc;">A interpretar el ajuste fino de un modelo para una tarea supervisada específica de clasificación textual.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Regla práctica que debes recordar</div>
    <div style="color:#f8fafc;">Un pipeline completo solo es sólido si cada parte anterior quedó bien resuelta y alineada con la tarea.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Contexto mínimo</div>
    <div style="color:#f8fafc;">Clasificar texto con fine-tuning implica adaptar un modelo preentrenado a nuevas etiquetas usando datos del dominio objetivo.</div>
  </div>
</div>

</div>

</div>

<hr style="border:0; border-top:1px solid #64748b; opacity:0.22; margin:36px 0 42px 0;">

In [ ]:
# Bloque 1 — Carga del modelo y del tokenizer
"""Este bloque carga un modelo preentrenado con una cabeza de clasificación y su tokenizer asociado.
Este bloque importa "AutoModelForSequenceClassification" y "AutoTokenizer" de "transformers" para cargar un modelo preentrenado 
y su tokenizador correspondiente.

Aplicamos la clase "AutoModelForSequenceClassification", acompañada del método ".from_pretrained()", a la variable "modelo" para cargar un modelo preentrenado.
El argumento "bert-base-uncased" especifica el modelo preentrenado que se desea utilizar.
El parámetro "num_labels=" especifica el número de etiquetas de clasificación que el modelo debe predecir.
La variable "modelo" almacena una instancia del modelo para clasificación de secuencias. La cabeza de clasificación se inicializa de nuevo y necesita entrenamiento.

Aplicamos la clase "AutoTokenizer", acompañada del método ".from_pretrained()", a la variable "tokenizador" para cargar un tokenizador preentrenado.
El argumento "bert-base-uncased" especifica el modelo preentrenado que se desea utilizar.
La variable "tokenizador" almacena una instancia del tokenizador preentrenado lista para ser utilizada en tareas de clasificación de secuencias.
Estos son los componentes fundamentales para realizar el fine-tuning de un modelo de clasificación de secuencias.
El modelo y el tokenizador están preentrenados para inglés, mientras que los textos del ejemplo están en español."""

from transformers import AutoModelForSequenceClassification, AutoTokenizer

modelo = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)
tokenizador = AutoTokenizer.from_pretrained("bert-base-uncased")

modelo, tokenizador # Mostramos el modelo y el tokenizador para verificar que se han cargado correctamente.


In [ ]:
# Bloque 2 — Construcción del dataset
"""Este bloque construye un dataset mínimo para la clasificación de texto.
Este bloque importa "Dataset" de "datasets" para construir un dataset compatible con Hugging Face.

Definimos un dataset mínimo con textos y etiquetas para la clasificación de texto.
La variable "textos" contiene una lista con los textos de entrada para la clasificación de texto.
La variable "etiquetas" contiene una lista de etiquetas correspondientes a cada uno de los textos de entrada.

Definimos el diccionario que contiene los textos y etiquetas a partir del cual se construye el dataset.
La variable "diccionario" almacena los textos y etiquetas que se utilizarán para construir el dataset.

Definimos la lista "lista_columnas" que contiene los nombres de las columnas que se incluirán en el dataset tokenizado.
La variable "lista_columnas" contiene los nombres de las columnas que se incluirán en el dataset tokenizado.

Aplicamos la clase "Dataset", acompañada del método ".from_dict()", a la variable "dataset" para crear el dataset.
El argumento "diccionario" es el diccionario definido previamente a partir del cual se construye el dataset.
La variable "dataset" contiene el dataset sin tokenizar, listo para ser tokenizado y utilizado en la clasificación de texto."""

from datasets import Dataset

textos = ["El producto es excelente.", "La experiencia fue terrible.", "El envío llegó rápido.", "La calidad es deficiente."]
etiquetas = [1, 0, 1, 0]
diccionario = {"texto": textos, "label": etiquetas}
lista_columnas = ["input_ids", "attention_mask", "label"]

dataset = Dataset.from_dict(diccionario)

dataset # Mostramos el dataset para verificar su contenido para la clasificación de texto.


In [ ]:
# Bloque 3 — Tokenización del dataset
"""Este bloque aplica tokenización al dataset.
Definimos la función "def tokenize_batch(batch)" para tokenizar cada lote de textos del dataset.
El parámetro "batch" es un diccionario que contiene los textos asociados a la clave "texto" del diccionario definido previamente.

Esta función retorna la llamada al tokenizador "tokenizador()" con los textos del lote, aplicando padding y truncado.
El argumento "batch["texto"]" contiene los textos del lote asociados a la clave "texto".
El parámetro "padding=" rellena las secuencias hasta la longitud de la más larga del lote.
El parámetro "truncation=" trunca las secuencias que exceden la longitud máxima del modelo.

Aplicamos el método ".map()" al dataset para tokenizar sus textos.
El argumento "tokenize_batch" es la función que se aplica a cada lote para realizar la tokenización.
El parámetro "batched=" indica que la función se aplicará a lotes de ejemplos en lugar de a ejemplos individuales.
La variable "dataset_tokenizado" almacena el dataset con los campos necesarios para el modelo.

Aplicamos el método ".set_format()" al dataset tokenizado para establecer el formato de salida.
El parámetro "type=" indica con el valor "torch" que los datos se devolverán como tensores de PyTorch.
El parámetro "columns=" especifica las columnas que se incluirán en la salida, en este caso, las columnas definidas en la lista "lista_columnas".
La variable "dataset_tokenizado" almacena ahora el dataset tokenizado en un formato compatible con PyTorch.
Esta operación permite que el dataset tokenizado se pueda utilizar directamente en modelos de PyTorch para entrenamiento o inferencia."""

def tokenize_batch(batch):
    return tokenizador(batch["texto"], padding=True, truncation=True)

dataset_tokenizado = dataset.map(tokenize_batch, batched=True)
dataset_tokenizado.set_format(type="torch", columns=lista_columnas)

dataset_tokenizado # Muestra el dataset tokenizado para verificar su contenido.


In [ ]:
# Bloque 4 — Configuración del entrenamiento
"""Este bloque define los parámetros de entrenamiento mediante TrainingArguments.
Este bloque importa "TrainingArguments" de "transformers" para definir los argumentos de entrenamiento.

Aplicamos la clase "TrainingArguments()" a la variable "argumentos" para definir los parámetros de entrenamiento.
El parámetro "output_dir=" especifica el directorio donde se guardarán los resultados de la clasificación, en este caso, "salida_clasificacion".
El parámetro "num_train_epochs=" indica el número de épocas para el entrenamiento, en este caso, 2.
El parámetro "per_device_train_batch_size=" define el tamaño de batch por dispositivo, en este caso, 2.
El parámetro "logging_steps=" establece cada cuántos pasos se registrará información de entrenamiento, en este caso, 1.

La variable "argumentos" almacena la configuración del proceso, incluyendo el número de épocas y el tamaño de batch.
Estos argumentos controlan el comportamiento del proceso de entrenamiento para la clasificación de texto."""

from transformers import TrainingArguments

argumentos = TrainingArguments(
    output_dir="salida_clasificacion",
    num_train_epochs=2,
    per_device_train_batch_size=2,
    logging_steps=1
)

argumentos # Muestra la configuración de entrenamiento definida en la variable "argumentos" para verificar los parámetros establecidos.


In [ ]:
# Bloque 5 — Definición del Trainer
"""Este bloque define el Trainer que coordina el proceso de fine-tuning.
Este bloque importa "Trainer" de "transformers" para coordinar el proceso de fine-tuning del modelo.

Aplicamos la clase "Trainer()" a la variable "entrenador" para coordinar el proceso de fine-tuning del modelo.
El parámetro "model=" especifica el modelo que se entrenará, en este caso, "modelo".
El parámetro "args=" recibe los argumentos de entrenamiento, en este caso, "argumentos".
El parámetro "train_dataset=" define el dataset de entrenamiento, en este caso, "dataset_tokenizado".

La variable "entrenador" almacena el objeto Trainer configurado con el modelo, los argumentos y el dataset tokenizado.
Trainer gestiona automáticamente el cálculo de pérdidas, la optimización y el registro de información de entrenamiento.
Este ejemplo no configura un conjunto de evaluación ni métricas de clasificación."""

from transformers import Trainer

entrenador = Trainer(
    model=modelo,
    args=argumentos,
    train_dataset=dataset_tokenizado
)

entrenador # Muestra el objeto Trainer configurado.


In [ ]:
# Bloque 6 — Ejecución del entrenamiento
"""Este bloque ejecuta el proceso de fine-tuning mediante el método train.
Aplicamos el método ".train()" a la variable "entrenador" para iniciar el proceso de fine-tuning del modelo.
La variable "resultado_entrenamiento" almacena la información generada durante el ajuste del modelo.
El ajuste con solo cuatro ejemplos es ilustrativo y no permite concluir que el modelo generalice."""

resultado_entrenamiento = entrenador.train()

resultado_entrenamiento # Muestra la información generada durante el ajuste del modelo.


In [ ]:
# Bloque 7 — Inferencia con el modelo ajustado
"""Este bloque realiza inferencia utilizando el modelo ajustado.
Definimos un texto de prueba para observar la salida del modelo.
La variable "entrada_prueba" contiene un texto para realizar inferencia con el modelo.

Asignamos la llamada a la instancia de la clase "tokenizador()" a la variable "ids_prueba" mediante la expresión "tokenizador(entrada_prueba, return_tensors="pt")".
El argumento "entrada_prueba" indica el texto que se desea tokenizar.
El parámetro "return_tensors=" indica el tipo de tensor que se desea obtener, en este caso, "pt" para PyTorch.
La variable "ids_prueba" almacena los tensores tokenizados que se pasan al modelo.

Asignamos la llamada a la instancia de la clase "modelo()" a la variable "salida_prueba" mediante la expresión "modelo(**ids_prueba)".
El argumento "**ids_prueba" indica los tensores tokenizados que se pasan al modelo.
Los asteriscos dobles "**" se utilizan para desempaquetar el diccionario "ids_prueba" y pasar sus elementos como argumentos al modelo.
La variable "salida_prueba" contiene la salida del modelo, incluidos los logits generados tras el entrenamiento.
Esta llamada no realiza una evaluación con etiquetas ni desactiva explícitamente los gradientes o el modo de entrenamiento."""

entrada_prueba = "El servicio fue muy bueno."

ids_prueba = tokenizador(entrada_prueba, return_tensors="pt")

salida_prueba = modelo(**ids_prueba)

salida_prueba # Mostramos la salida del modelo para la entrada de prueba.


In [ ]:
# Bloque 8 — Resultados finales
"""Este bloque organiza los resultados del proceso de fine-tuning para clasificación de texto.
La variable "resultado" almacena en forma de diccionario la salida del entrenamiento y los logits de la inferencia posterior."""

resultado = {
    "resultado_entrenamiento": resultado_entrenamiento,
    "logits_inferencia": salida_prueba.logits  # El atributo ".logits" contiene los logits de la salida del modelo.
}

resultado # Mostramos el diccionario que contiene los resultados del entrenamiento y los logits de la inferencia.


<div style="
    width:100%;
    max-width:1080px;
    margin:40px 0 24px 0;
    background:linear-gradient(180deg, #1f2937 0%, #0f172a 100%);
    border:1px solid #334155;
    border-radius:22px;
    box-shadow:0 18px 38px rgba(15,23,42,0.24);
    overflow:hidden;
    color:#e6edf7;
">

<div style="
    height:8px;
    background:linear-gradient(90deg, #0f766e 0%, #2dd4bf 100%);
"></div>

<div style="
    max-width:860px;
    padding:34px 38px 36px 38px;
    margin-left:0;
    margin-right:auto;
    font-family:'Aptos', 'Segoe UI Variable', 'Segoe UI', sans-serif;
    font-size:19px;
    line-height:1.78;
    color:#e6edf7;
">

<div style="margin-bottom:22px;">
<h2 style="margin:0 0 14px 0; font-size:30px; line-height:1.2; font-weight:700; color:#ffffff;">Conclusiones</h2>
<p style="margin:0; color:#cbd5e1; font-size:18px;">En este notebook has visto una aplicación completa de fine-tuning sobre una tarea de clasificación de texto.</p>
</div>

<div style="display:grid; gap:14px;">
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Qué conviene recordar</div>
    <div style="color:#f8fafc;">El valor del modelo preentrenado aparece de verdad cuando se integra con datos, configuración y evaluación coherentes.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Error frecuente a evitar</div>
    <div style="color:#f8fafc;">Atribuir todo el resultado al modelo y no al conjunto completo de decisiones del pipeline.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Siguiente paso natural</div>
    <div style="color:#f8fafc;">Detenerse en la Evaluación de los Modelos de Texto para dar continuidad directa al hilo del temario.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Conexión con el siguiente concepto</div>
    <div style="color:#f8fafc;">El siguiente notebook se centra en la Evaluación de los Modelos de Texto y amplía de forma natural lo que acabas de trabajar.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Siguiente notebook del temario</div>
    <div style="color:#f8fafc;">07_Evaluacion_de_modelos_de_texto/01_La_evaluacion_de_los_modelos_de_texto.ipynb</div>
  </div>
</div>

</div>

</div>

<hr style="border:0; border-top:1px solid #64748b; opacity:0.22; margin:42px 0 24px 0;">